## Imports

In [35]:
from dataclasses import asdict, dataclass, field, fields
from functools import cached_property
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy.optimize import linear_sum_assignment
from sklearn.cluster import DBSCAN

import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

## Parámetros

In [36]:
# Rutas de directorios
DATA_DIR = Path("data")
OUTPUT_DIR = Path("resultados")

# Parámetros de preprocesamiento
VOXEL_SIZE = 0.15   # Tamaño del vóxel en metros para el submuestreo

# Parámetros de detección (DBSCAN)
DBSCAN_EPS = 1.8
DBSCAN_MIN_SAMPLES = 8
MIN_POINTS = 80     # Mínimo de puntos que debe tener un clúster para ser considerado vehículo

# Parámetros de Tracking (Seguimiento)
TRACK_GATE = 5.0    # Distancia máxima de asociación en metros
TRACK_MAX_GAP = 1.0 # Tiempo máximo tolerado sin detección en segundos
TRACK_MIN_HITS = 3  # Detecciones consecutivas requeridas para confirmar un ID de vehículo

## Carga de Datos

In [37]:
@dataclass
class LidarFrame:
    index: int
    path: Path
    time: float
    points: np.ndarray


class LidarDataset:
    def __init__(self, data_dir, pattern='pointcloud_*.csv'):
        self.data_dir = Path(data_dir)
        self.files = sorted(self.data_dir.glob(pattern), key=self.timestamp_ns)
        if not self.files:
            raise ValueError(f'No se han encontrado frames en {self.data_dir}')
        self.start = self.timestamp_ns(self.files[0])

    def __len__(self):
        return len(self.files)

    def __iter__(self):
        for index, path in enumerate(self.files):
            yield LidarFrame(index, path, self.elapsed_s(path), self.load_points(path))

    def elapsed_s(self, path):
        return (self.timestamp_ns(path) - self.start) / 1e9

    @staticmethod
    def timestamp_ns(path):
        """Extrae el tiempo en nanosegundos del nombre del archivo.
        Los nanosegundos del nombre no están rellenados con ceros."""
        seconds, nanos = map(int, path.stem.split('_')[1:])
        return seconds * 1_000_000_000 + nanos

    @staticmethod
    def load_points(path):
        """Carga la nube de puntos desde un archivo CSV a un array de numpy filtrando valores no finitos."""
        points = np.loadtxt(path, delimiter=',', skiprows=1, ndmin=2)
        if points.size == 0:
            return np.empty((0, 3))
        if points.shape[1] != 3:
            raise ValueError(f'{path}: se esperaban las columnas x,y,z')
        return points[np.isfinite(points).all(axis=1)]

## Detección de Vehículos (DBSCAN)

In [38]:
@dataclass
class Detection:
    center: np.ndarray
    low: np.ndarray
    high: np.ndarray
    points: int

    @classmethod
    def from_cluster(cls, cluster):
        low, high = cluster.min(axis=0), cluster.max(axis=0)
        return cls((low[:2] + high[:2]) / 2, low, high, len(cluster))

    @property
    def size(self):
        return self.high - self.low


class VehicleDetector:
    def __init__(self):
        self.dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)

    def detect(self, points):
        objects = self._voxelize(points)
        labels = self._cluster(objects)
        candidates = (Detection.from_cluster(objects[labels == label])
                      for label in sorted(set(labels) - {-1}))
        return [d for d in candidates if self._is_vehicle(d)]

    def _voxelize(self, objects):
        if not len(objects):
            return objects
        # Un punto por vóxel evita que la densidad cercana domine DBSCAN.
        _, indices = np.unique(np.floor(objects / VOXEL_SIZE), axis=0,
                               return_index=True)
        return objects[indices]

    def _cluster(self, objects):
        if not len(objects):
            return np.empty(0, dtype=int)
        return self.dbscan.fit_predict(objects)

    @staticmethod
    def _is_vehicle(detection):
        # También admite vehículos parcialmente visibles en los bordes.
        size = detection.size
        return detection.points >= MIN_POINTS and size[1] >= .8 and size[2] >= .6

## Seguimiento de Vehículos (Tracking)

In [39]:
@dataclass
class Track:
    id: int
    center: np.ndarray
    time: float
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2))
    hits: int = 1
    streak: int = 1
    confirmed: bool = False

    def predict(self, time):
        return self.center + self.velocity * (time - self.time)

    def update(self, center, time, min_hits):
        dt = time - self.time
        if dt > 0:
            measured = (center - self.center) / dt
            self.velocity = .8 * self.velocity + .2 * measured
        self.center, self.time = center, time
        self.hits += 1
        self.streak += 1
        self.confirmed |= self.streak >= min_hits

    def miss(self):
        self.streak = 0


class Tracker:
    def __init__(self, gate=TRACK_GATE, max_gap=TRACK_MAX_GAP, min_hits=TRACK_MIN_HITS):
        self.gate = gate
        self.max_gap = max_gap
        self.min_hits = min_hits
        self.active = []
        self.tracks = []

    @property
    def confirmed_ids(self):
        return {track.id for track in self.tracks if track.confirmed}

    def update(self, detections, time):
        self.active = [t for t in self.active if time - t.time <= self.max_gap]
        assigned = self._associate(detections, time)
        matched = set(assigned.values())
        for track in self.active:
            if track.id not in matched:
                track.miss()
        for i, detection in enumerate(detections):
            if i not in assigned:
                assigned[i] = self._spawn(detection, time).id
        return assigned

    def _associate(self, detections, time):
        if not (self.active and detections):
            return {}
        predicted = np.array([t.predict(time) for t in self.active])
        centers = np.array([d.center for d in detections])
        distances = np.linalg.norm(predicted[:, None] - centers[None], axis=2)
        # Bloquear parejas imposibles ANTES de la asignación global.
        cost = np.where(distances <= self.gate, distances, 1e9)
        assigned = {}
        for row, col in zip(*linear_sum_assignment(cost)):
            if distances[row, col] <= self.gate:
                track = self.active[row]
                track.update(centers[col], time, self.min_hits)
                assigned[col] = track.id
        return assigned

    def _spawn(self, detection, time):
        track = Track(len(self.tracks) + 1, detection.center, time,
                      confirmed=self.min_hits <= 1)
        self.tracks.append(track)
        self.active.append(track)
        return track

## Resultados y Exportación

In [40]:
@dataclass
class Observation:
    frame: int
    tiempo_s: float
    id: int
    x: float
    y: float
    x_min: float
    y_min: float
    z_min: float
    x_max: float
    y_max: float
    z_max: float
    puntos: int

    @classmethod
    def from_detection(cls, frame, time, track_id, detection):
        return cls(frame, time, track_id, *detection.center,
                   *detection.low, *detection.high, detection.points)

    @property
    def low(self):
        return np.array([self.x_min, self.y_min, self.z_min])

    @property
    def high(self):
        return np.array([self.x_max, self.y_max, self.z_max])


@dataclass
class FrameRecord:
    frame: int
    archivo: str
    tiempo_s: float
    puntos_validos: int
    candidatos_dbscan: int
    vehiculos_actuales: int
    vehiculos_vistos: int
    vehiculos: int = 0
    ids: str = ''

    def set_vehicles(self, ids):
        self.vehiculos = len(ids)
        self.ids = ';'.join(map(str, ids))


@dataclass
class TrackingResult:
    frames: list
    observations: list
    clouds: list
    confirmed: set

    @classmethod
    def from_confirmed(cls, frames, observations, clouds, confirmed):
        result = cls(frames, [row for row in observations if row.id in confirmed],
                     clouds, confirmed)
        for record in frames:
            record.set_vehicles(sorted(row.id for row in result.by_frame.get(record.frame, [])))
        return result

    @cached_property
    def by_frame(self):
        grouped = {}
        for row in self.observations:
            grouped.setdefault(row.frame, []).append(row)
        return grouped


class ResultExporter:
    def __init__(self):
        self.output_dir = OUTPUT_DIR

    def export(self, result):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self._write_csv('conteo_frames.csv', result.frames, FrameRecord)
        self._write_csv('tracking.csv', result.observations, Observation)
        summary = self.summary(result)
        (self.output_dir / 'resumen.txt').write_text(summary, encoding='utf-8')
        return summary

    @staticmethod
    def summary(result):
        return (
            f'Frames: {len(result.frames)}\n'
            f'Duración: {result.frames[-1].tiempo_s:.3f} s\n'
            f'Vehículos únicos (IDs confirmados): {len(result.confirmed)}\n'
            f'Máximo de vehículos detectados en un frame: {max(f.vehiculos for f in result.frames)}\n'
            f'Vóxel: {VOXEL_SIZE} m\n'
            f'DBSCAN: eps={DBSCAN_EPS} m; min_samples={DBSCAN_MIN_SAMPLES}; min_points={MIN_POINTS}\n'
            f'Tracking: gate={TRACK_GATE} m; max_gap={TRACK_MAX_GAP} s; min_hits={TRACK_MIN_HITS}\n'
        )

    def _write_csv(self, name, rows, row_type):
        table = pd.DataFrame([asdict(row) for row in rows],
                             columns=[f.name for f in fields(row_type)])
        table.to_csv(self.output_dir / name, index=False, encoding='utf-8')

## Pipeline de Procesamiento

In [41]:
class TrackingPipeline:
    def __init__(self):
        self.dataset = LidarDataset(DATA_DIR)
        self.detector = VehicleDetector()
        self.tracker = Tracker()

    def run(self):
        frames, observations, clouds = [], [], []
        for frame in tqdm(self.dataset, desc='Procesando frames', unit='frame'):
            record, frame_observations = self._process(frame)
            frames.append(record)
            observations.extend(frame_observations)
            clouds.append(frame.points)
        return TrackingResult.from_confirmed(frames, observations, clouds,
                                             self.tracker.confirmed_ids)

    def _process(self, frame):
        detections = self.detector.detect(frame.points)
        ids = self.tracker.update(detections, frame.time)
        record = FrameRecord(frame.index, frame.path.name, frame.time,
                             len(frame.points), len(detections),
                             len(ids), len(self.tracker.tracks))
        observations = [Observation.from_detection(frame.index, frame.time, ids[i], detection)
                        for i, detection in enumerate(detections)]
        return record, observations

## Visualización 3D

In [42]:
class VehicleCloudExtractor:
    def __init__(self, max_points=6000):
        self.max_points = max_points

    def extract(self, points, rows):
        available = np.ones(len(points), dtype=bool)
        result = []
        for row in rows:
            inside = (((points >= row.low - VOXEL_SIZE) & (points <= row.high + VOXEL_SIZE))
                      .all(axis=1) & available)
            result.append(self.downsample(points[inside]))
            available[inside] = False
        return result

    def downsample(self, points):
        step = max(1, int(np.ceil(len(points) / self.max_points)))
        return points[::step].astype(np.float32, copy=True)

In [43]:
class TrackingVisualizer:
    PALETTE = px.colors.qualitative.Plotly
    TRACES_PER_VEHICLE = 3
    DEFAULT_BOUNDS = ((-50, 50), (-50, 50), (-5, 10))
    MARGINS = ((5, 5), (5, 5), (2, 5))

    def __init__(self, result, interval=50):
        self.frames = result.frames
        self.by_frame = result.by_frame
        self.interval = interval
        extractor = VehicleCloudExtractor()
        self.vehicle_clouds = [extractor.extract(points, self.by_frame.get(i, []))
                               for i, points in enumerate(result.clouds)]
        self.max_vehicles = max(map(len, self.by_frame.values()), default=0)
        self.fig = self._build_figure(self._bounds(result.observations))

    def show(self):
        last = len(self.frames) - 1
        play = widgets.Play(value=0, min=0, max=last, step=1, interval=self.interval,
                            description="Reproducir", show_repeat=False)
        slider = widgets.IntSlider(min=0, max=last, step=1, description='Frame:')
        widgets.jslink((play, 'value'), (slider, 'value'))
        slider.observe(lambda change: self.update_frame(change.new), names='value')
        display(widgets.HBox([play, slider]), self.fig)
        self.update_frame(0)

    def update_frame(self, index):
        rows = self.by_frame.get(index, [])
        with self.fig.batch_update():
            self.fig.layout.title = self._title(index)
            for slot in range(self.max_vehicles):
                start = slot * self.TRACES_PER_VEHICLE
                traces = self.fig.data[start:start + self.TRACES_PER_VEHICLE]
                if slot < len(rows):
                    self._draw_vehicle(traces, rows[slot], self.vehicle_clouds[index][slot])
                else:
                    self._clear(traces)

    @classmethod
    def _bounds(cls, observations):
        if not observations:
            return cls.DEFAULT_BOUNDS
        return tuple((min(getattr(row, f'{axis}_min') for row in observations) - below,
                      max(getattr(row, f'{axis}_max') for row in observations) + above)
                     for axis, (below, above) in zip('xyz', cls.MARGINS))

    def _build_figure(self, bounds):
        ranges = [high - low for low, high in bounds]
        largest = max(ranges)
        layout = dict(
            margin=dict(l=0, r=0, b=0, t=40),
            scene=dict(
                **{f'{axis}axis': dict(visible=False, range=list(limits), autorange=False)
                   for axis, limits in zip('xyz', bounds)},
                aspectmode='manual',
                aspectratio={axis: span / largest for axis, span in zip('xyz', ranges)},
                camera=dict(eye=dict(x=-1.2, y=-1.5, z=2.5))
            ),
            height=700,
            showlegend=False,
            uirevision='constant'
        )
        traces = [trace for _ in range(self.max_vehicles) for trace in self._vehicle_traces()]
        return go.FigureWidget(data=traces, layout=layout)

    @staticmethod
    def _vehicle_traces():
        return [
            go.Scatter3d(x=[], y=[], z=[], mode='markers', marker=dict(size=2, opacity=0.8), showlegend=False),
            go.Scatter3d(x=[], y=[], z=[], mode='lines', line=dict(width=4), showlegend=False, hoverinfo='none'),
            go.Scatter3d(x=[], y=[], z=[], mode='text', text=[], textfont=dict(size=16), showlegend=False),
        ]

    def _title(self, index):
        record = self.frames[index]
        return (
            f"Frame {index}/{len(self.frames) - 1} | "
            f"Num vehiculos actual: {record.vehiculos_actuales} | "
            f"Num vehiculos vistos: {record.vehiculos_vistos} | "
            f"Tiempo: {record.tiempo_s:.2f} s"
        )

    def _draw_vehicle(self, traces, row, points):
        cloud, box, label = traces
        color = self.PALETTE[(row.id - 1) % len(self.PALETTE)]

        cloud.x, cloud.y, cloud.z = points.T
        cloud.marker.color = color

        box.x, box.y, box.z = self.box_lines(row)
        box.line.color = color

        label.x, label.y, label.z = [row.x], [row.y], [row.z_max + 0.8]
        label.text = [f"<b>ID: {row.id}</b>"]
        label.textfont.color = color

    @staticmethod
    def _clear(traces):
        for trace in traces:
            trace.x, trace.y, trace.z = [], [], []
        traces[-1].text = []

    @staticmethod
    def box_lines(row):
        x0, y0, z0 = row.x_min, row.y_min, row.z_min
        x1, y1, z1 = row.x_max, row.y_max, row.z_max
        corners = [(x0, y0), (x1, y0), (x1, y1), (x0, y1), (x0, y0)]
        segments = [[(x, y, z) for x, y in corners] for z in (z0, z1)]
        segments += [[(x, y, z0), (x, y, z1)] for x, y in corners[:-1]]
        vertices = [vertex for segment in segments for vertex in segment + [(None, None, None)]][:-1]
        return [list(axis) for axis in zip(*vertices)]

## Procesamiento Principal y Resultados

In [44]:
result = TrackingPipeline().run()
print("\n" + ResultExporter().export(result))

print("Preparando visualización 3D interactiva...")
TrackingVisualizer(result).show()

Procesando frames:   0%|          | 0/369 [00:00<?, ?frame/s]


Frames: 369
Duración: 18.652 s
Vehículos únicos (IDs confirmados): 4
Máximo de vehículos detectados en un frame: 3
Vóxel: 0.15 m
DBSCAN: eps=1.8 m; min_samples=8; min_points=80
Tracking: gate=5.0 m; max_gap=1.0 s; min_hits=3

Preparando visualización 3D interactiva...


FigureWidget({
    'data': [{'marker': {'opacity': 0.8, 'size': 2},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter3d',
              'uid': 'b763205e-50f7-4250-99ec-3b359f5e3e45',
              'x': [],
              'y': [],
              'z': []},
             {'hoverinfo': 'none',
              'line': {'width': 4},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter3d',
              'uid': 'f9fcea01-d64a-434b-bc47-18a67a7aa73a',
              'x': [],
              'y': [],
              'z': []},
             {'mode': 'text',
              'showlegend': False,
              'text': [],
              'textfont': {'size': 16},
              'type': 'scatter3d',
              'uid': '6a211d4b-4fab-4ae5-bd1e-b29688b8be86',
              'x': [],
              'y': [],
              'z': []},
             {'marker': {'opacity': 0.8, 'size': 2},
              'mode': 'markers',
            